# 第 4 周练习：测试生成（Python）

## 练习目标

用 **OpenRouter**（OpenAI 兼容 API）+ **Gradio**：粘贴一段 Python 源码，让模型输出「原代码 + `test_...` 测试函数」。

## 和本课的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions | `client.chat.completions.create(...)` |
| system / user messages | `SYSTEM_PROMPT` + `user_prompt_for` |
| 多模型路由 | OpenRouter 上的 GPT / Claude 模型 id |
| Gradio UI | Dropdown 选模型，按钮触发生成 |

## 怎么跑

1. 在 `.env` 设置 `OPEN_ROUTER_API_KEY`
2. 依次运行单元格，最后 `demo.launch`
3. 粘贴代码 → 选模型 → Generate tests


In [ ]:
# ========== 导入：环境、正则、OpenAI 客户端、Gradio ==========

# 导入标准库 os：读 OPEN_ROUTER_API_KEY 等环境变量
import os
# 导入标准库 re：从模型回复里剥离 ```python 代码围栏
import re

# 从 dotenv 导入 load_dotenv：把 .env 密钥加载进进程环境
from dotenv import load_dotenv
# 从 openai 导入 OpenAI：通过自定义 base_url 走 OpenRouter
from openai import OpenAI
# 导入 gradio：搭「选模型 + 贴代码 + 生成测试」界面
import gradio as gr


In [ ]:
# ========== 加载密钥：确认 OpenRouter API Key 是否就绪 ==========

# 加载 .env；override=True 允许覆盖已有同名环境变量
load_dotenv(override=True)
# 读取 OpenRouter 密钥（变量名必须是 OPEN_ROUTER_API_KEY）
openrouter_key = os.getenv("OPEN_ROUTER_API_KEY")


# 有密钥时只打印前缀，避免把完整 key 打到日志里
if openrouter_key:
    print(f"OpenRouter key present (begins {openrouter_key[:8]}...)")
else:
    # 提示文案保持英文原样（依赖此字符串告知用户如何配置）
    print("Set OPEN_ROUTER_API_KEY in .env to use the app.")


In [ ]:
# ========== 客户端 + 模型表：一个 OpenRouter 客户端挂多条模型 ==========

# OpenRouter 的 OpenAI 兼容 API 基址（URL 勿改）
openrouter_url = "https://openrouter.ai/api/v1"
# 有 key 才创建客户端；否则 openai 为 None，后面 MODELS 变空字典
openai = OpenAI(api_key=openrouter_key, base_url=openrouter_url) if openrouter_key else None

# 展示名 → (客户端, OpenRouter 模型 id)；无客户端则整表为空
MODELS = {
    "OpenAI GPT-4o": (openai, "openai/gpt-4o"),
    "OpenAI GPT-4o-mini": (openai, "openai/gpt-4o-mini"),
    "OpenAI GPT-4o-mini (latest)": (openai, "openai/gpt-4o-mini-2024-07-18"),
    "Anthropic Claude 3.5 Sonnet": (openai, "anthropic/claude-3.5-sonnet"),
    "Anthropic Claude 3.5 Haiku": (openai, "anthropic/claude-3.5-haiku"),
    "Anthropic Claude 3 Opus": (openai, "anthropic/claude-3-opus"),
} if openai else {}

# 打印可选模型名列表；没有 key 时提示配置
print("Models:", list(MODELS.keys()) or "None (set OPEN_ROUTER_API_KEY)")


In [ ]:
# ========== Prompt：system 定规则，user 塞入待测源码（字符串勿改）==========

# 系统提示：要求输出可执行 Python（原代码 + test_ 函数），禁止 markdown 围栏
SYSTEM_PROMPT = """You are a Python engineer. Your task is to write Python tests for the given Python code.
Rules:
- Output raw Python only: first the original code (the module under test), then test functions (def test_...).
- Do not wrap the output in markdown or ``` blocks.
- Test every public function and class method with at least one simple test that should pass.
- Use assert for checks. No pytest required; plain Python with assert is fine.
"""

# 把用户粘贴的源码包进 user 消息模板
def user_prompt_for(python_source: str) -> str:
    return f"""Write Python tests for the following code. Include the original code at the top, then the test functions. Output only executable Python (no markdown, no ```).

Python code to test:

{python_source}
"""


In [ ]:
# ========== messages 组装：Chat Completions 标准两角色结构 ==========

# 输入待测源码，返回 [system, user] 消息列表
def messages_for(python_source: str):
    return [
        # system：全局规则与输出格式
        {"role": "system", "content": SYSTEM_PROMPT},
        # user：具体要测的代码（经 user_prompt_for 包装）
        {"role": "user", "content": user_prompt_for(python_source)},
    ]


In [ ]:
# ========== 生成：调模型 + 去掉可能的 markdown 围栏 ==========

# 若模型仍包了 ```python ... ```，用正则抽出中间正文
def extract_python(reply: str) -> str:
    """Remove markdown code fences if present; return raw Python."""
    # DOTALL 让 . 跨行；IGNORECASE 兼容 ```Python
    m = re.search(r"```(?:python)?\s*\n?(.*?)```", reply, re.DOTALL | re.IGNORECASE)
    if m:
        # 命中围栏：返回捕获组并去首尾空白
        return m.group(1).strip()
    # 无围栏：整段回复当作代码
    return reply.strip()


# 按界面展示名查 MODELS，发起 chat.completions，再清洗输出
def generate_tests(model_name: str, python_source: str) -> str:
    # 未知模型名：返回错误注释行（字符串保持原样）
    if model_name not in MODELS:
        return "# Error: unknown model"
    # 解包：OpenAI 兼容客户端 + 真实 model id
    client, model_id = MODELS[model_name]
    # 调用 Chat Completions；max_tokens 限制单次生成长度
    response = client.chat.completions.create(
        model=model_id,
        messages=messages_for(python_source),
        max_tokens=4096,
    )
    # 取第一条 choice 的文本；可能为 None，故 or ""
    reply = (response.choices[0].message.content or "").strip()
    # 去掉围栏后返回可粘贴的 Python
    return extract_python(reply)


In [ ]:
# ========== Gradio：校验输入 → 调 generate_tests → 展示结果 ==========

# UI 回调：做空输入/无模型检查，再委托 generate_tests
def ui_generate(model_name: str, python_source: str) -> str:
    # 没有源码：提示用户粘贴（文案保持英文）
    if not (python_source and python_source.strip()):
        return "(Enter Python code to test)"
    # 模型表为空或选项非法：提示配置密钥
    if not MODELS or model_name not in MODELS:
        return "(No model selected or set OPEN_ROUTER_API_KEY in .env)"
    # 去掉首尾空白后生成测试代码
    return generate_tests(model_name, python_source.strip())


# 创建 Blocks 应用
with gr.Blocks(title="Test Generation") as demo:
    # 标题与操作说明（界面英文保留）
    gr.Markdown("## Generate Python tests from your code")
    gr.Markdown("Paste your Python code, pick a model, then click **Generate tests**.")

    with gr.Row():
        # 模型下拉：无 key 时给占位选项
        model_dropdown = gr.Dropdown(
            choices=list(MODELS.keys()) if MODELS else ["(Set OPEN_ROUTER_API_KEY first)"],
            value=list(MODELS.keys())[0] if MODELS else None,
            label="Model",
        )
    # 多行文本：粘贴待测 Python
    python_input = gr.Textbox(
        label="Python code to test",
        lines=12,
        placeholder="def add(a, b): return a + b\n\ndef is_even(n): return n % 2 == 0",
    )
    # 触发生成的按钮
    run_btn = gr.Button("Generate tests")
    # 展示模型产出的测试代码
    generated_code = gr.Textbox(label="Generated test code", lines=14)
    # 点击：inputs → ui_generate → outputs
    run_btn.click(
        fn=ui_generate,
        inputs=[model_dropdown, python_input],
        outputs=generated_code,
    )

# 本地启动；share=False 不生成公网临时链接
demo.launch(share=False)
